In [ ]:
!mineru -p ./pdfs/energies-18-00645.pdf -o ./mineru/mineru_rag_output -b pipeline

2026-07-21 15:58:26.902 | INFO     | mineru.cli.client:run_orchestrated_cli:953 - Started local mineru-api at http://127.0.0.1:59905
2026-07-21 15:58:33.804 | INFO     | __main__:create_app:236 - Request concurrency limited to 3
Start MinerU FastAPI Service: http://127.0.0.1:59905
API documentation: http://127.0.0.1:59905/docs
INFO:     Started server process [299070]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:59905 (Press CTRL+C to quit)
2026-07-21 15:58:33.949 | INFO     | mineru.cli.client:run_planned_task:832 - Submitting batch 1/1 | 1 document, 33 pages in this batch | 33 pages total | task#1 [energies-18-00645]
2026-07-21 15:58:40.932 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:209 - Pipeline processing-window multi-file run. doc_count=1, total_pages=33, window_size=64, total_batches=1
2026-07-21 15:58:54.231 | INFO     | mineru.backend.pipeline.pipeline_analyze:d

In [ ]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import re

embeddings_model = OllamaEmbeddings(model="qwen3-embedding:4b")

def construct_vectorstore():
    
    text_splitter = RecursiveCharacterTextSplitter(
      
        chunk_size=2000,      
        chunk_overlap=150     
    )

    markdown_path = "./mineru/mineru_rag_output/energies-18-00645/auto/energies-18-00645.md"
    with open(markdown_path, "r", encoding="utf-8") as f:
        markdown_text = f.read()

    heading_pattern = re.compile(
        r"(?m)^(?P<heading>(?:#{1,6}\s*)?\d+(?:\.\d+)*\.\s+.+)$"
    ) 

    sections = []
    last_end = 0
    current_heading = None
    for match in heading_pattern.finditer(markdown_text):
        if current_heading is not None:
            body = markdown_text[last_end:match.start()].strip()
            if body:
                sections.append(Document(
                    page_content=f"{current_heading}\n{body}",
                    metadata={"heading": current_heading}
                ))
        current_heading = match.group("heading").strip()
        last_end = match.end()



    token_chunks = []
    for section in sections:
        token_chunks.extend(text_splitter.split_documents([section]))
    
    vector_store = Chroma.from_documents(
        documents=token_chunks, embedding=embeddings_model, persist_directory="./mineru/mineru_rag_chroma_demo")
    
    return vector_store


construct_vectorstore()

/home/zbta138a/miniconda3/envs/MinerU/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
vectordb = Chroma(
    persist_directory="./mineru/mineru_rag_chroma_demo", embedding_function=embeddings_model
)
print(vectordb.get()["metadatas"])

[{'heading': '## 1. Introduction'}, {'heading': '## 1.1. Literature Review'}, {'heading': '## 1.1. Literature Review'}, {'heading': '## 1.1. Literature Review'}, {'heading': '## 1.2. Resulting Problem and Contribution of This Paper'}, {'heading': '## 1.2. Resulting Problem and Contribution of This Paper'}, {'heading': '## 2. Materials and Methods'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.1. Linear Optimization Approach'}, {'heading': '## 2.2. Data Hierarchical Clustering Approach'}, {'heading': '## 2.2. Data Hierarchical Clustering Approa